# Recopilación de resultados de una sola ejecución

Al ejecutar nuestro DES, queremos **recopilar datos** que nos ayuden a analizar el rendimiento del sistema, como **tiempos de espera, utilización de recursos y longitud de las colas**.

¡Una herramienta como `simpy` le permite recopilar sus datos de manera flexible utilizando un enfoque que tenga sentido para usted! Algunas opciones son:

1. **Codificar un proceso de auditor/observador**.  Este proceso observará periódicamente el estado del sistema. Podemos usar esto para recopilar información sobre **el estado actual en el momento t**. Por ejemplo, cuántos pacientes están en cola y cuántos tienen una llamada en curso por hora del día.  

2. **Almacene métricas de proceso durante una ejecución y realice cálculos al final de una ejecución**. Por ejemplo, si desea calcular el tiempo medio de espera del paciente, almacene el tiempo de espera de cada paciente en una lista y calcule el tiempo medio al final de la serie.

3. **Realizar y auditar o calcular estadísticas de ejecución a medida que la simulación ejecuta un evento**.  Por ejemplo, cuando un paciente completa una llamada, podemos calcular una media acumulada de tiempos de espera y un total acumulado de operadores que atienden llamadas. Esta última medida se puede utilizar para calcular la utilización del servidor. También puede utilizar este enfoque para auditar la longitud de la cola, donde la longitud de la cola se registra cada vez que se realiza una solicitud de un recurso (y/o cuando se libera un recurso).

Este cuaderno proporciona un ejemplo de la **segunda estrategia**.

## 1. Importaciones

In [1]:
import simpy
import numpy as np
import itertools

## 2. Calcular el tiempo medio de espera

La segunda estrategia para la recopilación de resultados es almacenar una referencia a un valor cuantitativo (por ejemplo, tiempo de espera) durante la ejecución.  Una vez que se complete la ejecución, deberá incluir un procedimiento para calcular la métrica de interés.

* 😊 Una ventaja de esta estrategia es que es muy **simple**, captura todos los datos y tiene una sobrecarga computacional mínima durante la ejecución del modelo.
* 😢 Una posible desventaja es que, para simulaciones complejas, puedes terminar almacenando una **gran cantidad de datos en la memoria**. En estas circunstancias, puede valer la pena explorar estrategias impulsadas por eventos para reducir los requisitos de memoria.

![](./img/callcentre_waittime.png)

En nuestro ejemplo, haremos:

1. Cree una **lista** (`results['waiting_times']`) para almacenar el tiempo de espera de cada persona que llama
2. Cuando se ejecuta el modelo, **cada vez que una persona que llama ingresa al servicio**, la función `service()` agregará un `waiting_time` para la persona que llama a la lista.
3. Al final de la ejecución, recorreremos estas referencias y calcularemos **tiempo medio de espera**

### 2.1 Crear lista para almacenar tiempos de espera

Nuestra lista se almacenará dentro de un **diccionario** de Python que creamos llamado `results`. Esto significa que es sencillo agregar nuevas métricas (por ejemplo, utilización, longitud de la cola) en una fecha posterior.

El diccionario tiene alcance a nivel de cuaderno. Esto significa que **cualquier función o clase** en el cuaderno puede acceder y/o agregarse a la lista de acceso mediante la tecla `waiting_times`.

In [2]:
results = {}
results['waiting_times'] = []

### 2.2 Activar/desactivar declaraciones impresas

En todo el modelo, utilizamos declaraciones `print` para imprimir el progreso (por ejemplo, cada vez que un operador inicia y finaliza una llamada).

Para **habilitar/deshabilitar** la impresión, creamos una función auxiliar llamada `trace()` que envuelve `print`.  Luego podemos configurar una variable llamada `TRACE` para activar o desactivar estos mensajes de impresión.

In [3]:
def trace(msg):
    '''
    Turning printing of events on and off.
    
    Params:
    -------
    msg: str
        string to print to screen.
    '''
    if TRACE:
        print(msg)

### 2.3 Funciones de servicio y llegada

La única modificación que debemos hacer es la función `service`.  Agregaremos una línea de código para registrar el `waiting_time` de la persona que llama cuando ingresa al servicio.

```python
results['waiting_times'].append(waiting_time)
```

In [4]:
def service(identifier, operators, env, service_rng):
    '''
    Simulates the service process for a call operator

    1. request and wait for a call operator
    2. phone triage (triangular distribution)
    3. exit system
    
    Params:
    ------
    
    identifier: int 
        A unique identifier for this caller
        
    operators: simpy.Resource
        The pool of call operators that answer calls
        These are shared across resources.
        
    env: simpy.Environment
        The current environment the simulation is running in
        We use this to pause and restart the process after a delay.

    service_rng: numpy.random.Generator
        The random number generator used to sample service times
    
    '''
    # record the time that call entered the queue
    start_wait = env.now

    # request an operator
    with operators.request() as req:
        yield req

        # record the waiting time for call to be answered
        waiting_time = env.now - start_wait
        results['waiting_times'].append(waiting_time)
        
        trace(f'operator answered call {identifier} at ' \
              + f'{env.now:.3f}')

        # sample call duration.
        call_duration = service_rng.triangular(left=5.0, mode=7.0,
                                               right=10.0)
        
        # schedule process to begin again after call_duration
        yield env.timeout(call_duration)

        

        # print out information for patient.
        trace(f'call {identifier} ended {env.now:.3f}; ' \
              + f'waiting time was {waiting_time:.3f}')

In [5]:
def arrivals_generator(env, operators):
    '''
    IAT is exponentially distributed

    Parameters:
    ------
    env: simpy.Environment
        The simpy environment for the simulation

    operators: simpy.Resource
        the pool of call operators.
    '''
    # create the arrival process rng 
    arrival_rng = np.random.default_rng()
    
    # create the service rng that we pass to each service process created
    service_rng = np.random.default_rng()
    
    # use itertools as it provides an infinite loop 
    # with a counter variable that we can use for unique Ids
    for caller_count in itertools.count(start=1):

        # 100 calls per hour (sim time units = minutes). 
        inter_arrival_time = arrival_rng.exponential(60/100)
        yield env.timeout(inter_arrival_time)

        trace(f'call arrives at: {env.now:.3f}')

        # create a new simpy process for serving this caller.
        # we pass in the caller id, the operator resources, env, and the rng
        env.process(service(caller_count, operators, env, service_rng))

### 2.4 Realizar una sola ejecución del modelo

Podríamos conservar el código para ejecutar el modelo como un script. Sin embargo, es útil crear una nueva **función** llamada `single_run` que usamos para realizar una **replicación única** del modelo y devolver resultados. 

Si luego queremos ejecutar **múltiples replicaciones**, es solo un caso de ejecutar `single_run` en un **bucle**.

Agregamos una línea de código para encontrar los tiempos de espera medios de `results`.

In [6]:
def single_run(run_length, n_operators):
    '''
    Perform a single replication of the simulation model and 
    return the mean waiting time as a result.

    Parameters:
    ----------
    run_length: float
        The duration of the simulation run in minutes.

    n_operators: int
        The number of call operators to create as a resource

    Returns:
    -------
    mean_waiting_time: int
    '''
    # create simpy environment and operator resources
    env = simpy.Environment()
    operators = simpy.Resource(env, capacity=n_operators)
    
    env.process(arrivals_generator(env, operators))
    env.run(until=run_length)
    print(f'end of run. simulation clock time = {env.now}')
    
    # MODIFICATION calculate results on notebook level variables.
    mean_waiting_time = np.mean(results['waiting_times'])

    return mean_waiting_time

In [7]:
# reset data structure holding results
results = {}
results['waiting_times'] = []

# model parameters
RUN_LENGTH = 1000
N_OPERATORS = 13

# Turn off caller level results.
TRACE = False

mean_waiting_time = single_run(RUN_LENGTH, N_OPERATORS)
print("Simulation Complete")
print(f"Waiting time for call operators: {mean_waiting_time:.2f} minutes")

end of run. simulation clock time = 1000
Simulation Complete
Waiting time for call operators: 3.94 minutes
